# IOAI — 2024 First Stage Riddles (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/queries.txt'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-riddles/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 수수께끼 — 베이스라인 (설명↔정의 단순 겹침)

각 후보 단어의 위키낱말사전 **정의**와 수수께끼 설명의 (표제어화된) **토큰 겹침 개수**로 후보를 정렬한다. 가중치가 없어 흔한 단어에 휘둘려 MRR 이 낮다(≈0.12). 전체 원문/규칙은 Overview 탭 참고.

## 데이터 로드

In [ ]:
import re, math
from collections import defaultdict as dd, Counter
def tok(s): return re.findall(r"\w+", s.lower(), flags=re.UNICODE)
# 표제어(lemma) 사전
bases = {}
for x in open("data/lemmas.txt", encoding="utf-8"):
    p = x.lower().split()
    if len(p) == 2: bases[p[0]] = p[1]
def base(w): return bases.get(w, w)
# 후보 단어(명사)별 정의 토큰(표제어화, TF)
docs = {}
for x in open("data/definitions.txt", encoding="utf-8"):
    if "###" not in x: continue
    w, d = x.split("###", 1); L = w.split()
    if len(L) != 1: continue
    docs.setdefault(L[0], []).extend(base(t) for t in tok(d))
words = list(docs); df = dd(int)
for w in words:
    for t in set(docs[w]): df[t] += 1
inv = dd(list)
for w in words:
    for t, c in Counter(docs[w]).items(): inv[t].append((w, c))
queries = [tok(l) for l in open("data/queries.txt", encoding="utf-8").read().splitlines()]
print("candidates", len(words), "| queries", len(queries))

## answer_riddle — 단순 겹침

In [ ]:
def answer_riddle(riddle, K=20):
    qb = set(base(t) for t in riddle)
    sc = dd(int)
    for t in qb:
        for w, c in inv.get(t, []): sc[w] += 1          # 정의에 겹치는 표제어 수(가중치 없음)
    return [w for w, _ in sorted(sc.items(), key=lambda x: -x[1])[:K]]

## 순위 예측 → submission.txt

In [ ]:
ranked = [answer_riddle(set(q), K=20) for q in queries]
with open("submission.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(" ".join(r) for r in ranked))
print("saved submission.txt", len(ranked), "lines")

IDF/BM25 로 흔한 단어를 눌러주면 크게 개선된다 — 모범답안 참고.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.txt']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)